annotation meta for tree figures

In [4]:
import pandas as pd

# 1. メタデータの読み込み (TSV) と抽出
df_meta = pd.read_csv("/Users/izumikida/FCoV-evolution/FCoV_spike_seqs/FCoV1_S_dates_final_meta.tsv", sep='\t')
df_meta = df_meta[["Accession", "biotype"]]

# 2. スコアデータの読み込み (CSV) とリネーム
df_scores = pd.read_csv("/Users/izumikida/FCoV-evolution/FCoV_spike_mutation_combination_effect/phen_scores_dates.csv")
df_scores = df_scores.rename(columns={"Unnamed: 0": "Accession"})

# 3. マージ
df_all = pd.merge(df_meta, df_scores, on="Accession")
df_all = df_all.drop(columns= 'SNP_818')
df_all

,Accession,biotype,SNP_791,SNP_809,SNP_811,SNP_813,SNP_814,SNP_826,SNP_1056,SNP_1072,SNP_1074,SNP_1078,SNP_1129,SNP_1133,SNP_1135,SNP_1160,SNP_1212,SNP_1363,SNP_1368
0,PQ416775.1,FIPV,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,0
1,PX315748.1,FIPV,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0
2,PX315749.1,FIPV,1,1,0,0,1,0,1,1,0,0,0,0,0,0,0,1,0
3,PX315750.1,FIPV,1,1,0,0,1,0,1,1,0,0,0,0,0,0,0,1,0
4,PX315751.1,FIPV,0,0,0,0,1,0,1,1,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,FJ917521.1,FIPV,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0
116,FJ917522.1,FECV,1,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0
117,FJ917535.1,FIPV,1,1,1,0,0,0,1,1,0,0,1,0,0,0,0,0,1
118,FJ917523.1,FIPV,1,1,1,1,0,0,1,1,0,0,0,1,0,0,0,0,0


In [5]:
from Bio import SeqIO
import pandas as pd
import re

# 1. FASTAファイルを読み込み（辞書形式：Accession -> Sequence）
fasta_path = "/Users/izumikida/FCoV-evolution/FCoV_spike_seqs/FCoV1_S_all_seq_cleaned_cds_linsi_curated_fullllength_fix.aa_dates.fasta"
fasta_dict = {rec.id: str(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")}

# 2. df_all から SNP_ で始まるカラム名を抽出
snp_cols = [col for col in df_all.columns if col.startswith("SNP_")]

# 3. アミノ酸情報を格納するための新しいDataFrameを作成
df_all_aa = df_all.copy()

# 4. 各SNPカラムについて、対応するアミノ酸をFASTAから取得
for col in snp_cols:
    # カラム名から数字（ポジション）を抽出 (例: SNP_811 -> 811)
    pos = int(re.search(r'\d+', col).group())
    
    # Pythonのインデックスは0から始まるため、ポジションから1を引く
    # 各Accession（行）に対してアミノ酸を取得
    df_all_aa[f"AA_{pos}"] = df_all["Accession"].map(
        lambda acc: fasta_dict[acc][pos-1] if acc in fasta_dict else None
    )
df_all_aa

,Accession,biotype,SNP_791,SNP_809,SNP_811,SNP_813,SNP_814,SNP_826,SNP_1056,SNP_1072,...,AA_1072,AA_1074,AA_1078,AA_1129,AA_1133,AA_1135,AA_1160,AA_1212,AA_1363,AA_1368
0,PQ416775.1,FIPV,0,0,0,0,0,0,1,1,...,L,S,A,A,I,D,Q,I,P,P
1,PX315748.1,FIPV,0,0,1,0,0,0,1,0,...,M,A,A,A,I,D,Q,T,P,P
2,PX315749.1,FIPV,1,1,0,0,1,0,1,1,...,L,S,A,A,I,D,Q,T,-,P
3,PX315750.1,FIPV,1,1,0,0,1,0,1,1,...,L,S,A,A,I,D,Q,T,-,P
4,PX315751.1,FIPV,0,0,0,0,1,0,1,1,...,L,S,A,S,I,D,Q,T,P,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,FJ917521.1,FIPV,1,1,1,0,0,0,1,1,...,L,S,A,A,I,D,Q,T,P,P
116,FJ917522.1,FECV,1,1,1,0,0,0,1,1,...,L,S,A,A,I,D,Q,T,P,P
117,FJ917535.1,FIPV,1,1,1,0,0,0,1,1,...,L,S,A,S,I,D,Q,T,P,S
118,FJ917523.1,FIPV,1,1,1,1,0,0,1,1,...,L,S,A,A,V,D,Q,T,P,P


In [6]:
from Bio import SeqIO
import re

# 1. FASTAを辞書形式で読み込み (Accession -> Sequence)
fasta_path = "/Users/izumikida/FCoV-evolution/FCoV_spike_seqs/FCoV1_S_all_seq_cleaned_cds_linsi_curated_fullllength_fix.aa_dates.fasta"
fasta_dict = {rec.id: str(rec.seq) for rec in SeqIO.parse(fasta_path, "fasta")}

# 2. 上書き対象のカラム（SNP_で始まるもの）を特定
snp_cols = [col for col in df_all.columns if col.startswith("SNP_")]

# 3. 各SNPカラムをアミノ酸記号で上書き
for col in snp_cols:
    # カラム名からポジション番号を抽出 (例: SNP_811 -> 811)
    pos = int(re.search(r'\d+', col).group())
    
    # 各行の Accession に基づいてアミノ酸を取得（1-indexedを0-indexedに調整）
    # FASTAにAccessionがない場合は元の値を保持、またはNoneにする処理
    df_all[col] = df_all["Accession"].map(
        lambda acc: fasta_dict[acc][pos-1] if acc in fasta_dict else None
    )

df_all

,Accession,biotype,SNP_791,SNP_809,SNP_811,SNP_813,SNP_814,SNP_826,SNP_1056,SNP_1072,SNP_1074,SNP_1078,SNP_1129,SNP_1133,SNP_1135,SNP_1160,SNP_1212,SNP_1363,SNP_1368
0,PQ416775.1,FIPV,A,S,R,R,R,T,G,L,S,A,A,I,D,Q,I,P,P
1,PX315748.1,FIPV,A,S,S,R,R,T,Q,M,A,A,A,I,D,Q,T,P,P
2,PX315749.1,FIPV,V,P,R,R,K,T,A,L,S,A,A,I,D,Q,T,-,P
3,PX315750.1,FIPV,V,P,R,R,K,T,A,L,S,A,A,I,D,Q,T,-,P
4,PX315751.1,FIPV,A,S,R,R,M,T,Q,L,S,A,S,I,D,Q,T,P,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,FJ917521.1,FIPV,V,A,K,R,R,T,H,L,S,A,A,I,D,Q,T,P,P
116,FJ917522.1,FECV,V,A,K,R,R,T,H,L,S,A,A,I,D,Q,T,P,P
117,FJ917535.1,FIPV,V,A,K,R,R,T,H,L,S,A,S,I,D,Q,T,P,S
118,FJ917523.1,FIPV,V,A,K,L,R,T,G,L,S,A,A,V,D,Q,T,P,P


In [7]:
df_all.to_csv("tree_annotation_biotype_SNP.csv", index=False)